In [ ]:
import pandas as pd
from db_utils.database import build_engine, read_sql

engine = build_engine()

raw_data = read_sql(
    engine,
    "SELECT date, id, value FROM macro_data WHERE id = 'rate_gs10' ORDER BY date",
)
derived_data = read_sql(
    engine,
    """
    SELECT date, id, value
    FROM shiller_derived_view
    WHERE id IN ('r_sp_earn', 'r_sp_price', 'r_sp_div')
    ORDER BY date
    """,
)

data = pd.concat([raw_data, derived_data], axis=0).set_index(['date', 'id'])['value'].unstack()

future_earnings = read_sql(
    engine,
    """
    SELECT
        date,
        CASE
            WHEN COUNT(*) OVER (
                ORDER BY date
                ROWS BETWEEN 1 FOLLOWING AND 120 FOLLOWING
            ) = 120 THEN
                AVG(value) OVER (
                    ORDER BY date
                    ROWS BETWEEN 1 FOLLOWING AND 120 FOLLOWING
                )
            ELSE NULL
        END AS value
    FROM shiller_derived_view
    WHERE id = 'r_sp_earn'
    ORDER BY date
    """,
).set_index('date').rename(columns={'value': 'future_earnings'})

engine.dispose()

data.head()

In [ ]:
future_earnings.head()

In [ ]:
future_earnings

In [ ]:
y = future_earnings['future_earnings'].div(data['r_sp_price']).dropna().rename('rel_future_earnings')

In [ ]:
y.plot()

In [ ]:
type(data['rate_gs10'])